In [13]:
import sys
from pathlib import Path

CWD = Path(__name__).resolve().parent
sys.path.append(CWD)

DATASET_FILE = CWD / "result.json"
# OUT_CHECKPOINT_FILE = CWD / "leanrag_checkpoint.json"
# USE_CHECKPOINT_AS_CACHE = True # prefer data in the checkpoint file over re-computing?

secrets = CWD / "secrets.env"

if not secrets.is_file():
    raise ValueError(f"secrets file at '{secrets}' does not exist")

from dotenv import load_dotenv
load_dotenv(secrets)

# (START WITH G0 IN MEMGRAPH) ...

True

In [14]:
# TESTING FUNCTIONS
import utils.mg_driver as mg_driver
await mg_driver.init()

layer = 0
entity_descs = await mg_driver.get_entity_descs_for_layer(layer)
n_entities = len(entity_descs)
print(f"entities in layer {layer}: {n_entities}")
print(entity_descs[0] if entity_descs else "")

entities in layer 0: 620
{'key': 'detecting_aimbot_usage', 'desc': 'Identifying aimbot usage in video games, such as Minecraft, is essential for maintaining fair play and ensuring the integrity of the gaming experience.'}


In [21]:
# test embed batches
import asyncio
from utils import batched
import litellm
from tqdm import tqdm

EMBED_MODEL = "bedrock/amazon.titan-embed-text-v2:0"
ENTITY_BATCH_SIZE = 32
MAX_PARALLEL_EMBED = 8

class AsyncList:
    """write-protected list via asyncio lock"""
    def __init__(self):
        self._list = []
        self._lock = asyncio.Lock()

    def __getitem__(self, index:int):
        """only read from this. elements are not write-protected"""
        return self._list[index]

    def __len__(self): return len(self._list)

    def __str__(self): return str(self._list)

    async def extend(self, rows):
        async with self._lock:
            self._list.extend(rows)
    
    async def append(self, itm):
        async with self._lock:
            self._list.append(itm)

    async def get_list(self):
        """only perform read operations from this"""
        return self._list

async def batch_embed_descriptions(batch, acc:AsyncList, errors:AsyncList, embed_sem:asyncio.Semaphore, pbar, pbar_lock):
    try:
        async with embed_sem:
            resp = await litellm.aembedding(model=EMBED_MODEL, input=[e['desc'] for e in batch])
        batch_embed = resp['data']
        # unpack batch to entity_key -> description pairs
        rows = [
            {"key": batch[emb.index]["key"], "desc_embed": emb.embedding}
            for emb in sorted(batch_embed, key=lambda e: e.index)
        ]
        await acc.extend(rows)
        
        #progress bar
        async with pbar_lock:
            pbar.update(len(batch))
    except Exception as e:
        await errors.append({'batch': batch, 'error': str(e)})

async def embed_all_entity_descriptions(entity_descs:list, batch_size:int, max_parallel:int):

    embed_sem = asyncio.Semaphore(max_parallel)
    acc = AsyncList()
    errors = AsyncList()
    pbar, pbar_lock = tqdm(total=n_entities, desc="Entity description embeddings"), asyncio.Lock()

    batch_embed_tasks=[]
    for batch in batched(entity_descs, batch_size):
        batch_embed_tasks.append(batch_embed_descriptions(batch, acc, errors, embed_sem, pbar, pbar_lock))

    await asyncio.gather(*batch_embed_tasks, return_exceptions=True)
    pbar.close()
    print(f"created embeddings for entity descriptions for #{n_entities} entities")
    print(f"Errors ({len(errors)}):")
    print(errors)

    return acc

_result = await embed_all_entity_descriptions(entity_descs, ENTITY_BATCH_SIZE, MAX_PARALLEL_EMBED)
print(_result[0])


















Entity description embeddings: 100%|██████████| 620/620 [00:16<00:00, 37.04it/s]

created embeddings for entity descriptions for #620 entities
Errors (0):
[]
{'key': 'detecting_aimbot_usage', 'desc_embed': [-0.012098509818315506, 0.028729669749736786, 0.024201950058341026, 0.0221940316259861, 0.06762626022100449, -0.07353492081165314, 0.0587860532104969, -0.021348772570490837, 0.028079744428396225, 0.037589557468891144, 0.02854708395898342, 0.006871395278722048, 0.07502076029777527, -0.022780971601605415, 0.030224217101931572, -0.030342329293489456, -0.008812671527266502, -0.010864662937819958, 0.06007416918873787, -0.012648176401853561, 0.021608369424939156, 0.005937222391366959, -0.0009109668899327517, 0.06549134850502014, -0.024952517822384834, -0.002392977476119995, 0.03471551090478897, -0.005823319312185049, 0.0304552111774683, 0.0009907837957143784, -0.006069825496524572, 0.0684310644865036, -0.024530908092856407, -0.028663624078035355, -0.04379033297300339, 0.021384472027420998, -0.0433877594769001, 0.017231103032827377, 0.016352692618966103, 0.01669185049831

In [ ]:
import numpy as np
X = np.array([e['desc_embed'] for e in entity_desc_embeds])
print(X.shape)

(620, 1024)


In [ ]:
#test using gmm to partition
from math import ceil
from sklearn.mixture import GaussianMixture
import numpy as np
from collections import defaultdict

CLUSTER_SIZE = 20 # hyperparameter
n_components = ceil(n_entities/CLUSTER_SIZE)
X = np.asarray([e["desc_embed"] for e in entity_desc_embeds], dtype=np.float32)

gmm = GaussianMixture(
        n_components= n_components,
        covariance_type="diag",
        random_state=0,
        reg_covar=1e-6,
        max_iter=300,
        n_init=3
)
gmm.fit(X)

responsibilities = gmm.predict_proba(X) # probability that embedding i belongs to component k
labels = responsibilities.argmax(axis=1) # most likely component for embedding

# cluster using hard labels
# maps cluster # -> list of indexes for entities in entity_embed_descs that belong to the cluster
clusters = {k:[] for k in range(n_components)}
for i, k in enumerate(labels):
    clusters[int(k)].append(i)

{0: [11, 17, 18, 206, 207, 208, 211, 212, 217, 436], 1: [21, 26, 27, 30, 53, 55, 56, 58, 98, 106, 107, 113, 116, 117, 119, 120, 126, 128, 132, 138, 139, 160, 167, 169, 175, 234, 251, 257, 261, 262, 267, 394, 462, 464, 465, 470, 471], 2: [36, 99, 100, 101, 102, 103, 108, 109, 118, 131, 402, 409, 412, 415, 423, 434], 3: [34, 122, 123, 129, 144, 182, 189, 229, 230, 362, 364, 365, 366, 367, 371, 372, 374, 475, 481], 4: [31, 121, 168, 340, 341, 342, 344, 384, 385, 386, 388, 389, 390, 391, 393, 404, 410, 468, 469, 473], 5: [291, 292, 293, 294, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 309, 445, 446, 447, 448, 449, 450, 451, 452, 453], 6: [61, 133, 248, 249, 250, 252, 258, 259, 260, 264, 266, 345, 346, 348, 349, 350, 376, 377, 378, 381, 387, 392, 463], 7: [5, 9, 32, 52, 205, 401, 504, 505, 506, 508, 509, 510, 512, 513, 518, 519, 548, 549, 550, 551], 8: [265, 329, 330, 331, 335, 338, 339, 352, 379, 380, 382, 383, 477, 478, 492, 493], 9: [3, 4, 7, 8, 14, 83, 84, 88, 94, 95, 14

In [ ]:
# putting it all together into a function that can be run to aggregate a full layer
#TODO: add entity_type during extraction
import dspy
from typing import TypedDict
from utils import normalize_from_name

CLUSTER_SIZE = 20 # hyperparameter , soft
TAU = ... # todo - 
gmm = GaussianMixture(
    n_components= n_components,
    covariance_type="diag",
    random_state=0,
    reg_covar=1e-6,
    max_iter=300,
    n_init=3
)

class Finding(TypedDict):
    summary:str
    explanation:str

class AggEntity(TypedDict):
    key:str
    name:str
    description:str
    findings_id:str

class CreateAggregateNode(dspy.Signature):
    """
You are an expert in concept synthesis. Your task is to identify a meaningful aggregate entity from a set of related entities and extract structured insights based solely on provided evidence.

## SKILLS
- Abstraction and naming of collective concepts based on entity types
- Structured summarization and typology recognition
- Comparative analysis across multiple entities
- Strict grounding to provided data (no hallucinated content)

## GOALS
- Derive a meaningful aggregate entity that broadly represents the given entity set
- The aggregate entity name must not match any single entity in the set
- Provide an accurate and concise description of the aggregate entity reflecting shared characteristics
- Extract 5-10 structured findings about the entity set based on grounded evidence

## Rules
- Grounding Rule: All content must be based solely on the provided entity set — no external assumptions
- Naming Rule: The aggregate entity name must not be identical to any single entity; it should reflect a composite structure, function, or theme
- Each finding must include a concise summary and a detailed explanation
- Avoid adding speculative or unsupported interpretations

## Workflows
1. Review the list of entities, focusing on types, descriptions, and relational structure
2. Synthesize a generalized name that best represents the full entity set
3. Write a clear, evidence-based description of the aggregate entity
4. Extract and elaborate on key findings, emphasizing structure, purpose, and interconnections
    """
    input_text:str = dspy.InputField(desc="The input list of entities and their relations")
    entity_name:str = dspy.OutputField(desc="<name>")
    entity_description:str = dspy.OutputField(desc="<brief description summarizing the shared traits and structure>")
    findings:list[Finding] = dspy.OutputField(desc=(
            "JSON array of objects. "
            "Each object MUST have keys: 'summary' (string) and 'explanation' (string). "
            "Example: "
            "["
            "{'summary': '...', 'explanation': '...'}, "
            "{'summary': '...', 'explanation': '...'}"
            "]"))

async def gen_aggregate_entity(cluster:list, findings_map) -> AggEntity:
    """ aggregate generation function F(entity), creates parent node for cluster"""
    # collect entities in the cluster, and relations between all of them
    # assemble entity + relation rows for the LLM context:
    
    input_rows = []
    input_rows.append ("ENTITIES: entity_name, entity_description, entity_degree\n") # header
    for i, entity in enumerate(cluster):
        # form the csv row with entity information
        input_rows.append(f"{i}: {entity['name']}, {entity['desc']}, {entity['degree']}")
    input_rows.append("")
    
    intra_rels = await mg_driver.get_intra_cluster_relations(cluster)
    input_rows.append ("RELATIONS: source_entity, target_entity, relation_description") # header
    for i, rel in intra_rels:
        input_rows.append(f"{i}: {rel['source_entity']}, {rel['target_entity']}, {rel['relation_description']}")
    
    try:
        agg_resp = await dspy.predict(CreateAggregateNode(input_text="\n".join(input_rows)))
    except Exception as e:
        print(f"Erorr generating aggregate entity .. {e}")
        return None

    # create entry in the findings map
    agg_key = normalize_from_name(agg_resp['name'])
    findings_map[agg_key] = agg_resp['findings']
    return AggEntity(key=agg_key, name=agg_resp['name'], description=agg_resp['description'])

def gen_aggregate_rel(cj, ck, conn_strength, tau) :
    """ F(rel): when high enough conn strength between cj and ck, generate a relation.
      (else create with concatenation)"""
    if conn_strength > tau:
        ...
    else:
        ...

    return rel

findings_map : dict[str, list[Finding]] = {}
async def aggregate_layer(layer:int):
    # 1. collect all entities in the layer
    entities = await mg_driver.get_entities_for_layer(layer)
    n_entities = len(entities)
    
    # 2. batch embed them
    entity_desc_embeds = await embed_all_entity_descriptions([e['desc'] for e in entities], ENTITY_BATCH_SIZE, MAX_PARALLEL_EMBED)
    
    # 3. Feed embeds into GMM to partition into clusters
    n_components = ceil(n_entities/CLUSTER_SIZE)
    X = np.asarray([e["desc_embed"] for e in entity_desc_embeds], dtype=np.float32)
    gmm.fit(X)

    responsibilities = gmm.predict_proba(X) # probability that embedding i belongs to component k
    labels = responsibilities.argmax(axis=1) # most likely component for embedding

    # cluster using hard labels
    # maps cluster # -> list of entities
    clusters = {k:[] for k in range(n_components)}
    for i, k in enumerate(labels):
        clusters[int(k)].append(entities[i])

    # iterate clusters, 
    aggregates = []
    for cluster in clusters.keys():
        new_parent:AggEntity = await gen_aggregate_entity(cluster, findings_map) #also populates entry in findings_map
        aggregates.append(new_parent)
        # insert it into memgraph, creating link between children and new parent
        await mg_driver.create_aggregate_entity(new_parent, [e['key'] for e in entities])

    tau = ...
    # 4.all cluster aggregates are inserted, now create inter-cluster relations between all
    for cj in aggregates:
        for ck in aggregates:
            if ck == ck: continue
            # collect all relations between entities in cj and entities in ck
            rel_cj_ck = ... # memgraph cypher
            conn_strength = len(rel_cj_ck)
            
            agg_rel = gen_aggregate_rel(cj,ck, conn_strength, tau)

            # create relation in memgraph
            ...

In [22]:
print(entity_desc_embeds[0])

{'key': 'detecting_aimbot_usage', 'desc_embed': [-0.012098509818315506, 0.028729669749736786, 0.024201950058341026, 0.0221940316259861, 0.06762626022100449, -0.07353492081165314, 0.0587860532104969, -0.021348772570490837, 0.028079744428396225, 0.037589557468891144, 0.02854708395898342, 0.006871395278722048, 0.07502076029777527, -0.022780971601605415, 0.030224217101931572, -0.030342329293489456, -0.008812671527266502, -0.010864662937819958, 0.06007416918873787, -0.012648176401853561, 0.021608369424939156, 0.005937222391366959, -0.0009109668899327517, 0.06549134850502014, -0.024952517822384834, -0.002392977476119995, 0.03471551090478897, -0.005823319312185049, 0.0304552111774683, 0.0009907837957143784, -0.006069825496524572, 0.0684310644865036, -0.024530908092856407, -0.028663624078035355, -0.04379033297300339, 0.021384472027420998, -0.0433877594769001, 0.017231103032827377, 0.016352692618966103, 0.016691850498318672, -0.0349711999297142, 0.020207788795232773, -0.0011358397314324975, -0.

In [ ]:
# aggregate

# FOR EACH LAYER:
# 1. collect all e_descs=[entity.desc for entity in layer]
# 2. create the set of embeddings e_embeds = [embed(desc) for desc in e_descs]
# 3. Feed e_embeds into GMM to partition the layer into clusters:list

from sklearn.mixture import GaussianMixture
from dataclasses import dataclass

@dataclass
class AggNode:
    name:str
    desc:str
    child_entities:list[str]

def gen_aggregate_entity(cluster:Cluster) -> AggNode:
    """ aggregate generation function (F), creates parent node for cluster"""
# - get all relations between entities in this cluster in this layer : Question : isnt 'in this layer' implied?
# - ask an LLM to generate a name and description given these relations : Question : what exactly do we feed the LLMs
# -> OUTPUT = (new_parent_name, new_parent_description)
    return AggNode(name=agg_name, desc=agg_desc)

def gen_aggregate_rel(cj:Cluster, ck:Cluster):
    """ F(rel)"""
    ...
def concat_rels(relations) -> str:
    ...

new_aggregates = []
for cluster in clusters:
    new_agg_node:AggNode = gen_aggregate(cluster)
    new_agg_node.child_entities = cluster.entities
    new_aggregates.append(new_agg_node)
    # insert the agg parent into the graph, and
    # link each entity in the cluster to this new parent

TAU = ...
# inter-cluster connection:
for cj in new_aggregates:
    for ck in new_aggregates:
        if cj == ck: continue

        # collect all relations between entities in cj and entities in ck
        rel_cj_ck = ... # memgraph cypher
        conn_strength = len(rel_cj_ck)
        
        if conn_strength > TAU:
            agg_rel = gen_aggregate_rel(cj,ck)
        else:
            agg_rel = concat_rels(rel_cj_ck)

        # create relation in memgraph

from utils.mg_driver import get_entity_descs_for_layer
import litellm

gmm = GaussianMixture(...)
async def process_layer(layer:int):
    """Recursive aggregation from G0 -> LeanRAG KG"""
    # collect all entity descriptions from the layer
    entity_descs = await get_entity_descs_for_layer(layer)
    entity_desc_embeds = [] 
    # create embeddings for entity descriptions
    for batch in batched(entity_descs.items()):
        batch_embeds = await litellm.embedding(input=[e_desc for _,e_desc in batch])
        entity_desc_embeds.append(batch_embeds)

    # parititon with GMM        
    partitions = gmm.fit_predict(entity_desc_embeds)
    for cluster in partitions:
        # fetch entities that are part of this cluster
        ...
